# Two Follow-ups: Adaptive-Rank Merge + Blocks 7-11 Structural Diagnostic

**Part A** implements the effective-rank-proportional merge motivated by the finding that dolly uses only 77.3% of its rank-16 budget while metamath/codealpaca use ~89-90% — allocates rank per-layer proportional to measured effective rank instead of an equal split, merges, uploads, and evaluates end-to-end.

**Part B** digs into blocks 7-11 specifically — the zone where GSM8K and HumanEval showed their widest cross-method CKA spread, with `dare` flipping sign between the two tasks (highest CKA + worst GSM8K, highest CKA + best HumanEval, same blocks). Breaks that zone down by module type (attention vs MLP) to localize where the divergence actually originates.

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth peft
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Load Adapters + Base Model

In [2]:
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from unsloth import FastLanguageModel
import json
import torch

def get_lora_deltas(repo_name: str, hf_username: str = "Srishtik", lora_alpha: int = None, r: int = None) -> dict:
    repo_id = f"{hf_username}/{repo_name}"
    config_path = hf_hub_download(repo_id=repo_id, filename="adapter_config.json")
    cfg = json.load(open(config_path))
    actual_r     = r if r is not None else cfg.get("r")
    actual_alpha = lora_alpha if lora_alpha is not None else cfg.get("lora_alpha")
    path = hf_hub_download(repo_id=repo_id, filename="adapter_model.safetensors")
    adapter_weights = load_file(path)
    scale = actual_alpha / actual_r
    layers = {}
    for key, val in adapter_weights.items():
        if "lora_A" in key:
            base_key = key.replace("lora_A.default.weight", "").replace("lora_A.weight", "")
            layers.setdefault(base_key, {})["A"] = val.float()
        elif "lora_B" in key:
            base_key = key.replace("lora_B.default.weight", "").replace("lora_B.weight", "")
            layers.setdefault(base_key, {})["B"] = val.float()
    deltas = {}
    for base_key, mats in layers.items():
        if "A" in mats and "B" in mats:
            deltas[base_key] = scale * (mats["B"] @ mats["A"])
    print(f"  {repo_name:<35} r={actual_r}  alpha={actual_alpha}  scale={scale:.4f}")
    return deltas

print("Loading adapter deltas:")
dolly_deltas      = get_lora_deltas("qwen3-trained-on-dolly-15k")
metamath_deltas   = get_lora_deltas("qwen3-trained-on-metamath-15k")
codealpaca_deltas = get_lora_deltas("qwen3-trained-on-code-alpaca-18k")

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-0.6B",
    max_seq_length = 2048,
    load_in_4bit   = False,
    dtype          = torch.float16,
)
base_sd = base_model.state_dict()
del base_model
torch.cuda.empty_cache()

def normalize_delta_keys(deltas: dict) -> dict:
    normalized = {}
    for k, v in deltas.items():
        new_key = k.replace("base_model.model.", "").rstrip(".") + ".weight"
        normalized[new_key] = v
    return normalized

dolly_deltas      = normalize_delta_keys(dolly_deltas)
metamath_deltas   = normalize_delta_keys(metamath_deltas)
codealpaca_deltas = normalize_delta_keys(codealpaca_deltas)
deltas = [dolly_deltas, metamath_deltas, codealpaca_deltas]
adapter_names = ["dolly", "metamath", "codealpaca"]

print(f"\nOverlap with base_sd: {len(set(dolly_deltas.keys()) & set(base_sd.keys()))}")  # expect 196

def apply_delta_to_base(base_state_dict: dict, delta_state_dict: dict) -> dict:
    merged = {}
    for key, base_val in base_state_dict.items():
        if key in delta_state_dict:
            delta_val = delta_state_dict[key].to(base_val.device)
            merged[key] = (base_val.float() + delta_val).to(base_val.dtype)
        else:
            merged[key] = base_val
    return merged


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading adapter deltas:


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

  qwen3-trained-on-dolly-15k          r=16  alpha=32  scale=2.0000


adapter_config.json: 0.00B [00:00, ?B/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

  qwen3-trained-on-code-alpaca-18k    r=16  alpha=32  scale=2.0000
==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


Overlap with base_sd: 196


## Part A: Effective-Rank-Proportional Adaptive Merge

### The Method

In [3]:
import torch
def effective_rank(matrix: torch.Tensor) -> float:
    """Entropy-based effective rank (Roy & Vetterli, 2007)."""
    S = torch.linalg.svdvals(matrix.float())
    S = S[S > 1e-10]
    if len(S) == 0:
        return 0.0
    p = S / S.sum()
    entropy = -(p * torch.log(p)).sum()
    return torch.exp(entropy).item()


def adaptive_rank_merge(deltas: list, total_rank_budget: int = 24, min_rank: int = 2):
    """
    Effective-rank-proportional merge: instead of truncating every adapter to
    an EQUAL share of a shared rank budget (what BWSum currently does — an
    equal split regardless of how much rank each adapter actually uses),
    allocate rank PER LAYER, proportional to each adapter's own measured
    effective rank at that specific layer.

    Motivation: measured directly on these three adapters, dolly's mean
    effective rank is 12.37 (77.3% of nominal 16) while metamath and
    codealpaca sit at 14.36/14.24 (~89-90%) — dolly is using meaningfully
    less of its rank-16 budget than the other two. An equal split shortchanges
    the adapters that actually need more capacity and over-allocates to the
    one that doesn't.

    total_rank_budget=24 is a middle ground between BWSum's shared rank-16
    (aggressive compression) and a full rank-48 (sum of individual ranks,
    no compression at all) — chosen to keep the merged model's footprint
    modest while giving meaningfully more room than 16.
    """
    keys = set.intersection(*[set(d.keys()) for d in deltas])
    merged = {}
    allocation_log = {}

    for key in keys:
        eranks = [effective_rank(d[key]) for d in deltas]
        total_erank = sum(eranks)

        raw_alloc = [max(min_rank, round(total_rank_budget * (e / total_erank))) for e in eranks]
        diff = total_rank_budget - sum(raw_alloc)
        if diff != 0:
            idx = raw_alloc.index(max(raw_alloc))
            raw_alloc[idx] += diff

        allocation_log[key] = {"eranks": eranks, "alloc": raw_alloc}

        combined = torch.zeros_like(deltas[0][key])
        for d, r in zip(deltas, raw_alloc):
            U, S, Vh = torch.linalg.svd(d[key].float(), full_matrices=False)
            r_eff = min(r, len(S))
            truncated = U[:, :r_eff] @ torch.diag(S[:r_eff]) @ Vh[:r_eff, :]
            combined = combined + truncated
        merged[key] = combined

    return merged, allocation_log


merged_delta_adaptive, allocation_log = adaptive_rank_merge(deltas, total_rank_budget=24, min_rank=2)

# ── Report the aggregate allocation across all layers ──
import numpy as np
mean_alloc = np.mean([log["alloc"] for log in allocation_log.values()], axis=0)
print(f"Mean rank allocation across all {len(allocation_log)} layers (total budget = 24):")
for name, alloc in zip(adapter_names, mean_alloc):
    print(f"  {name:<12} {alloc:.2f}  ({100*alloc/24:.1f}% of budget)")
print(f"\n(equal split would be {24/3:.2f} each, {100/3:.1f}%)")


Mean rank allocation across all 196 layers (total budget = 24):
  dolly        7.19  (30.0% of budget)
  metamath     8.47  (35.3% of budget)
  codealpaca   8.34  (34.8% of budget)

(equal split would be 8.00 each, 33.3%)


### Apply + Upload

In [ ]:
import os
import torch
from unsloth import FastLanguageModel

merged_sd_adaptive = apply_delta_to_base(base_sd, merged_delta_adaptive)
print(f"Merged state dict ready: {len(merged_sd_adaptive)} keys")


def upload_merged_model(merged_sd, repo_name, tokenizer, hf_token,
                         base_repo="unsloth/Qwen3-0.6B", max_seq_length=2048,
                         dtype=torch.float16, push_to_hub=True):
    print(f"[upload] Preparing model → {repo_name}")
    model, _ = FastLanguageModel.from_pretrained(
        model_name=base_repo, max_seq_length=max_seq_length, load_in_4bit=False, dtype=dtype,
    )
    target_dtype = next(model.parameters()).dtype
    cast_sd = {k: v.to(target_dtype) if v.is_floating_point() else v for k, v in merged_sd.items()}
    missing, unexpected = model.load_state_dict(cast_sd, strict=False)
    if missing:
        print(f"  [warn] Missing keys   : {len(missing)}  (e.g. {missing[:3]})")
    if unexpected:
        print(f"  [warn] Unexpected keys: {len(unexpected)} (e.g. {unexpected[:3]})")
    model.eval()
    if push_to_hub:
        commit_info = model.push_to_hub(repo_name, token=hf_token, private=False)
        tokenizer.push_to_hub(repo_name, token=hf_token, private=False)
        print(f"  [hub] Pushed → https://huggingface.co/{repo_name}")
        print(f"  [hub] Commit info: {commit_info}")
    del model, cast_sd
    torch.cuda.empty_cache()
    return repo_name


# Use Kaggle Secrets / Colab secrets rather than pasting a token directly:
# from kaggle_secrets import UserSecretsClient
# HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
import os
HF_TOKEN = key
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not set. The previous version of this notebook had a live token "
        "hardcoded here -- that token should be revoked at "
        "https://huggingface.co/settings/tokens if it hasn't been already. "
        "Set HF_TOKEN via an environment variable or a secrets manager instead."
    )

ADAPTIVE_REPO = "Srishtik/Qwen3-0.6B-adaptive-rank-3-adapters-merged-2"
uploaded_adaptive_repo = upload_merged_model(
    merged_sd = merged_sd_adaptive,
    repo_name = ADAPTIVE_REPO,
    tokenizer = tokenizer,
    hf_token  = HF_TOKEN,
)


Merged state dict ready: 311 keys
[upload] Preparing model → Srishtik/Qwen3-0.6B-adaptive-rank-3-adapters-merged-2
==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/526 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpjj4ianti/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


  [hub] Pushed → https://huggingface.co/Srishtik/Qwen3-0.6B-adaptive-rank-3-adapters-merged-2
  [hub] Commit info: None


### Evaluate

In [5]:
import torch
import gc
import re
import math
import multiprocessing
import contextlib
import io
from datasets import load_dataset
from tqdm import tqdm
from unsloth import FastLanguageModel

def build_chat_prompt(tokenizer, user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

def prep_tokenizer_for_generation(tokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

# ── GSM8K ──
def extract_gsm8k_answer(text: str) -> str:
    hash_match = re.findall(r"####\s*(-?[\d,]+\.?\d*)", text)
    if hash_match:
        return hash_match[-1].replace(",", "").strip()
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].replace(",", "").strip()
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return text.strip()

def format_gsm8k_prompt(question: str) -> str:
    return (f"Solve the following math problem. Show your reasoning and put "
            f"your final numeric answer after '#### '.\n\nQuestion: {question}")

def evaluate_gsm8k(model, tokenizer, model_name="model", num_samples=200, batch_size=4,
                    max_new_tokens=320, device="cuda"):
    print(f"\n{'─'*60}\n[GSM8K] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("gsm8k", "main", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    preds, labels = [], []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [gsm8k]"):
        batch = dataset[i : i + batch_size]
        questions    = batch["question"]
        true_answers = [extract_gsm8k_answer(a) for a in batch["answer"]]
        prompts      = [build_chat_prompt(tokenizer, format_gsm8k_prompt(q)) for q in questions]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=512, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.3)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            preds.append(extract_gsm8k_answer(generated))
            labels.append(true_answers[j])
    per_sample_exact = [int(p.strip() == l.strip()) for p, l in zip(preds, labels)]
    exact_match = round(sum(per_sample_exact) / len(per_sample_exact), 4)
    print(f"  Exact Match: {exact_match:.4f}")
    return {"repo_id": model_name, "exact_match": exact_match, "num_samples": len(per_sample_exact),
            "per_sample_exact": per_sample_exact}

# ── HumanEval ──
def extract_code(generated: str, problem_prompt: str, entry_point: str) -> str:
    text = generated.strip()
    fence = re.search(r"```(?:python)?\s*\n?(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    if f"def {entry_point}" in text:
        return text
    return problem_prompt + "\n" + text

def _unsafe_execute(program: str, result_list, timeout: int):
    import signal
    def handler(signum, frame):
        raise TimeoutError("execution timed out")
    try:
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(timeout)
        exec_globals = {}
        with contextlib.redirect_stdout(io.StringIO()):
            exec(program, exec_globals)
        signal.alarm(0)
        result_list.append("passed")
    except Exception as e:
        result_list.append(f"failed: {type(e).__name__}: {e}")

def check_correctness(problem: dict, completion_code: str, timeout: int = 5) -> bool:
    program = completion_code + "\n" + problem["test"] + f"\ncheck({problem['entry_point']})\n"
    manager = multiprocessing.Manager()
    result_list = manager.list()
    p = multiprocessing.Process(target=_unsafe_execute, args=(program, result_list, timeout))
    p.start()
    p.join(timeout=timeout + 1)
    if p.is_alive():
        p.kill(); p.join()
    if not result_list:
        result_list.append("failed: timeout")
    return result_list[0] == "passed"

def format_humaneval_prompt(problem_prompt: str) -> str:
    return ("Complete the following Python function. Return ONLY the complete "
            "function code (including the signature), with no explanations and "
            f"no markdown formatting.\n\n{problem_prompt}")

def evaluate_humaneval(model, tokenizer, model_name="model", num_samples=164, batch_size=4,
                        max_new_tokens=384, device="cuda"):
    print(f"\n{'─'*60}\n[HumanEval] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("openai/openai_humaneval", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    per_sample_pass = []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [humaneval]"):
        batch = dataset[i : i + batch_size]
        problem_prompts = batch["prompt"]
        entry_points    = batch["entry_point"]
        prompts = [build_chat_prompt(tokenizer, format_humaneval_prompt(p)) for p in problem_prompts]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=768, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.1)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            code = extract_code(generated, problem_prompts[j], entry_points[j])
            problem = {"prompt": problem_prompts[j], "test": batch["test"][j], "entry_point": entry_points[j]}
            per_sample_pass.append(int(check_correctness(problem, code, timeout=5)))
    pass_at_1 = round(sum(per_sample_pass) / len(per_sample_pass), 4)
    print(f"  pass@1: {pass_at_1:.4f}")
    return {"repo_id": model_name, "pass_at_1": pass_at_1, "num_samples": len(per_sample_pass),
            "per_sample_pass": per_sample_pass}

# ── Dolly-15k perplexity ──
def format_dolly_prompt(instruction: str, context: str) -> str:
    if context:
        return f"Instruction: {instruction}\nContext: {context}\nResponse:"
    return f"Instruction: {instruction}\nResponse:"

def evaluate_dolly_perplexity(model, tokenizer, model_name="model", num_samples=200,
                               max_length=512, device="cuda"):
    print(f"\n{'─'*60}\n[Dolly-PPL] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))
    per_sample_nll, per_sample_ppl = [], []
    for ex in tqdm(dataset, desc=f"{model_name} [dolly-ppl]"):
        prompt = format_dolly_prompt(ex["instruction"], ex.get("context", ""))
        response = ex["response"]
        if not response.strip():
            continue
        full_text = prompt + " " + response
        prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        full_ids   = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        if full_ids.shape[1] <= prompt_ids.shape[1]:
            continue
        labels = full_ids.clone()
        labels[:, : prompt_ids.shape[1]] = -100
        with torch.no_grad():
            out = model(full_ids, labels=labels)
        nll = out.loss.item()
        per_sample_nll.append(nll)
        per_sample_ppl.append(math.exp(nll))
    result = {"repo_id": model_name, "perplexity": round(sum(per_sample_ppl) / len(per_sample_ppl), 4),
              "mean_nll": round(sum(per_sample_nll) / len(per_sample_nll), 4),
              "num_samples": len(per_sample_ppl), "per_sample_nll": per_sample_nll}
    print(f"  Perplexity: {result['perplexity']:.4f}  (mean NLL: {result['mean_nll']:.4f})")
    return result


In [6]:
import gc

eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = uploaded_adaptive_repo,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = torch.float16,
)
FastLanguageModel.for_inference(eval_model)

adaptive_gsm8k     = evaluate_gsm8k(eval_model, eval_tokenizer, model_name=uploaded_adaptive_repo,
                                     num_samples=200, batch_size=4)
adaptive_humaneval = evaluate_humaneval(eval_model, eval_tokenizer, model_name=uploaded_adaptive_repo,
                                         num_samples=164, batch_size=4)
adaptive_dolly     = evaluate_dolly_perplexity(eval_model, eval_tokenizer, model_name=uploaded_adaptive_repo,
                                                num_samples=200)

del eval_model, eval_tokenizer
gc.collect()
torch.cuda.empty_cache()

known_results = {
    "linear": {"gsm8k": 0.1100, "humaneval": 0.2012, "dolly_ppl": 17.8096},
    "svd"   : {"gsm8k": 0.1550, "humaneval": 0.2073, "dolly_ppl": 20.5070},
    "ties"  : {"gsm8k": 0.0550, "humaneval": 0.2073, "dolly_ppl": 16.7428},
    "dare"  : {"gsm8k": 0.0400, "humaneval": 0.2378, "dolly_ppl": 17.2781},
    "codealpaca_adapter (specialist)": {"gsm8k": 0.0850, "humaneval": 0.1707, "dolly_ppl": 38.3633},
    "metamath_adapter (specialist)"  : {"gsm8k": 0.2200, "humaneval": 0.1220, "dolly_ppl": 28.6676},
    "dolly_adapter (specialist)"     : {"gsm8k": 0.0250, "humaneval": 0.1646, "dolly_ppl": 12.6278},
}

print(f"\n{'═'*72}")
print(f"{'Model':<38}{'GSM8K':>10}{'HumanEval':>12}{'Dolly PPL':>12}")
print("-" * 72)
for name, r in known_results.items():
    print(f"{name:<38}{r['gsm8k']:>10.4f}{r['humaneval']:>12.4f}{r['dolly_ppl']:>12.4f}")
print("-" * 72)
print(f"{'adaptive-rank-merge (NEW)':<38}{adaptive_gsm8k['exact_match']:>10.4f}"
      f"{adaptive_humaneval['pass_at_1']:>12.4f}{adaptive_dolly['perplexity']:>12.4f}")
print("=" * 72)


==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K] Evaluating: Srishtik/Qwen3-0.6B-adaptive-rank-3-adapters-merged-2
────────────────────────────────────────────────────────────


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Srishtik/Qwen3-0.6B-adaptive-rank-3-adapters-merged-2 [dolly-ppl]: 100%|██████████| 200/200 [00:25<00:00,  7.86it/s]


  Perplexity: 19.1178  (mean NLL: 2.1636)

════════════════════════════════════════════════════════════════════════
Model                                      GSM8K   HumanEval   Dolly PPL
------------------------------------------------------------------------
linear                                    0.1100      0.2012     17.8096
svd                                       0.1550      0.2073     20.5070
ties                                      0.0550      0.2073     16.7428
dare                                      0.0400      0.2378     17.2781
codealpaca_adapter (specialist)           0.0850      0.1707     38.3633
metamath_adapter (specialist)             0.2200      0.1220     28.6676
dolly_adapter (specialist)                0.0250      0.1646     12.6278
------------------------------------------------------------------------
adaptive-rank-merge (NEW)                 0.0000      0.1585     19.1178


## Diagnostic: Raw GSM8K Completions (adaptive-rank model)

GSM8K exact-match collapsed to 0.0000 for the adaptive-rank merge -- worse than every
other method including DARE (0.0400). Before concluding the effective-rank hypothesis
itself is wrong, check whether the model is producing coherent-but-incorrect math or
degenerate/malformed output (empty strings, repeated tokens, broken chat formatting).


In [7]:
import gc

diag_model, diag_tok = FastLanguageModel.from_pretrained(
    model_name     = uploaded_adaptive_repo,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = torch.float16,
)
FastLanguageModel.for_inference(diag_model)
prep_tokenizer_for_generation(diag_tok)

from datasets import load_dataset

gsm8k_sample = load_dataset("gsm8k", "main", split="test").select(range(8))
prompts = [build_chat_prompt(diag_tok, format_gsm8k_prompt(q)) for q in gsm8k_sample["question"]]
inputs = diag_tok(prompts, return_tensors="pt", padding=True, truncation=True,
                   max_length=512, add_special_tokens=False).to("cuda")

with torch.no_grad():
    outputs = diag_model.generate(**inputs, max_new_tokens=320, do_sample=False,
                                   pad_token_id=diag_tok.pad_token_id, eos_token_id=diag_tok.eos_token_id,
                                   repetition_penalty=1.3)

input_len = inputs["input_ids"].shape[1]
for i, out in enumerate(outputs):
    generated = diag_tok.decode(out[input_len:], skip_special_tokens=True)
    true_ans  = extract_gsm8k_answer(gsm8k_sample["answer"][i])
    pred_ans  = extract_gsm8k_answer(generated)
    print(f"{'='*70}\nQ{i}: {gsm8k_sample['question'][i][:120]}...\n{'-'*70}")
    print(f"GENERATED:\n{generated}\n")
    print(f"extracted pred: {pred_ans!r}   true: {true_ans!r}   match: {pred_ans.strip() == true_ans.strip()}\n")

del diag_model, diag_tok
gc.collect()
torch.cuda.empty_cache()


==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Q0: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every da...
----------------------------------------------------------------------
GENERATED:
Janet's ducks lays 16 eggs per day, so they are laying a total of 48 eggs each night.
She eats three eggs from the ducks during breakfast, leaving them to be laid out on average of 35 eggs per night.
In addition, she also needs to bake muffin for his friends which leaves him being laid out an additional 90 eggs per night.
So he is left with a total of 127 eggs per night that can be sold by selling it at the farmer's market.
He makes 2 dollar worth of money per fresh egg when baking muffins for his friend, therefore he earns 2 * (127 - 9) = 208 dolars per week while baking muffins for his friend.
Therefore, he earned 208 + (3 x 208) = #### dolars per month or year as a result of eating eggs for breakfast and baking muffens for his friends.
The above calculation shows how many dola

## Part B: Blocks 7-11 Structural Diagnostic

Independent of Part A — operates on the raw adapter deltas directly. Breaks down effective rank and pairwise cosine similarity by module type, restricted to the blocks flagged by the per-layer CKA spread analysis.

In [8]:
import torch
import re
from collections import defaultdict

CRITICAL_BLOCKS = list(range(7, 12))  # blocks 7-11, the zone flagged by the CKA spread analysis
MODULE_TYPES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def parse_block_and_module(key: str):
    m = re.search(r"layers\.(\d+)\.", key)
    block = int(m.group(1)) if m else None
    module = next((t for t in MODULE_TYPES if t in key), None)
    return block, module


def cosine_sim(a: torch.Tensor, b: torch.Tensor) -> float:
    a, b = a.flatten().float(), b.flatten().float()
    na, nb = torch.norm(a), torch.norm(b)
    if na < 1e-8 or nb < 1e-8:
        return float("nan")
    return torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()


# ── Group keys by module type, restricted to blocks 7-11 ──
keys_by_module = defaultdict(list)
for key in set(dolly_deltas.keys()) & set(metamath_deltas.keys()) & set(codealpaca_deltas.keys()):
    block, module = parse_block_and_module(key)
    if block in CRITICAL_BLOCKS and module is not None:
        keys_by_module[module].append(key)

print(f"Blocks {CRITICAL_BLOCKS[0]}-{CRITICAL_BLOCKS[-1]} — per-module-type breakdown\n")
print(f"{'Module':<12}{'n_layers':>10}{'dolly erank':>13}{'meta erank':>12}{'code erank':>12}"
      f"{'dolly-meta cos':>16}{'dolly-code cos':>16}{'meta-code cos':>15}")
print("-" * 110)

adapters = {"dolly": dolly_deltas, "metamath": metamath_deltas, "codealpaca": codealpaca_deltas}
module_summary = {}

for module in MODULE_TYPES:
    ks = keys_by_module[module]
    if not ks:
        continue
    eranks = {name: [] for name in adapters}
    cos_dm, cos_dc, cos_mc = [], [], []
    for key in ks:
        for name, d in adapters.items():
            eranks[name].append(effective_rank(d[key]))
        cos_dm.append(cosine_sim(dolly_deltas[key], metamath_deltas[key]))
        cos_dc.append(cosine_sim(dolly_deltas[key], codealpaca_deltas[key]))
        cos_mc.append(cosine_sim(metamath_deltas[key], codealpaca_deltas[key]))

    mean_erank = {name: sum(v) / len(v) for name, v in eranks.items()}
    module_summary[module] = {
        "eranks": mean_erank,
        "cos_dm": sum(cos_dm) / len(cos_dm),
        "cos_dc": sum(cos_dc) / len(cos_dc),
        "cos_mc": sum(cos_mc) / len(cos_mc),
    }

    print(f"{module:<12}{len(ks):>10}{mean_erank['dolly']:>13.3f}{mean_erank['metamath']:>12.3f}"
          f"{mean_erank['codealpaca']:>12.3f}{module_summary[module]['cos_dm']:>+16.4f}"
          f"{module_summary[module]['cos_dc']:>+16.4f}{module_summary[module]['cos_mc']:>+15.4f}")

print("\nAttention (q/k/v/o) vs MLP (gate/up/down) aggregate comparison:")
attn_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
mlp_modules  = ["gate_proj", "up_proj", "down_proj"]

for group_name, group in [("Attention", attn_modules), ("MLP", mlp_modules)]:
    present = [m for m in group if m in module_summary]
    if not present:
        continue
    avg_mc_cos = sum(module_summary[m]["cos_mc"] for m in present) / len(present)
    avg_meta_erank = sum(module_summary[m]["eranks"]["metamath"] for m in present) / len(present)
    avg_code_erank = sum(module_summary[m]["eranks"]["codealpaca"] for m in present) / len(present)
    print(f"  {group_name:<10} mean metamath-codealpaca cosine: {avg_mc_cos:+.4f}   "
          f"mean erank (meta/code): {avg_meta_erank:.2f}/{avg_code_erank:.2f}")

print("\nIf metamath-codealpaca cosine similarity or effective rank differs sharply between")
print("attention and MLP modules specifically in blocks 7-11, that localizes WHERE the")
print("math/code sign-flip (dare: best for HumanEval, worst for GSM8K) actually originates.")


Blocks 7-11 — per-module-type breakdown

Module        n_layers  dolly erank  meta erank  code erank  dolly-meta cos  dolly-code cos  meta-code cos
--------------------------------------------------------------------------------------------------------------
q_proj               5       12.910      14.515      14.857         +0.0324         +0.0264        +0.0189
k_proj               5       12.332      14.237      14.409         +0.0374         +0.0251        +0.0286
v_proj               5       12.512      14.157      14.288         +0.0727         +0.0539        +0.0572
o_proj               5       13.667      14.571      14.509         +0.0322         +0.0209        +0.0098
gate_proj            5        9.222      14.117      14.709         +0.0149         +0.0070        +0.0083
up_proj              5       12.183      15.001      15.106         +0.0299         +0.0162        +0.0150
down_proj            5       14.014      14.913      14.427         +0.0270         +0.0127        

## Follow-up: Rank-Budget Sweep

`total_rank_budget=24` was one arbitrary point on a spectrum between BWSum's rank-16
(aggressive) and rank-48 (no compression). Before writing off effective-rank-proportional
allocation entirely, sweep the budget to see whether 24 specifically was too aggressive,
or whether the approach underperforms across the whole range. Runs GSM8K only, at a
reduced sample count, to keep this affordable -- confirm any promising budget with a
full 200-sample eval (and HumanEval/Dolly) before drawing conclusions.

Evaluates locally (no push_to_hub) at full precision, mirroring the exact state-dict-loading
step `upload_merged_model` performs before pushing -- loading directly into a 4-bit-quantized
model here would silently mismatch keys instead of raising an error.


In [9]:
import gc
import numpy as np

def build_and_evaluate_budget(total_rank_budget, min_rank=2, gsm8k_samples=40):
    merged_delta, alloc_log = adaptive_rank_merge(deltas, total_rank_budget=total_rank_budget, min_rank=min_rank)
    merged_sd = apply_delta_to_base(base_sd, merged_delta)

    mean_alloc = np.mean([log["alloc"] for log in alloc_log.values()], axis=0)
    alloc_str = ", ".join(f"{name}={a:.2f}" for name, a in zip(adapter_names, mean_alloc))
    print(f"budget={total_rank_budget}: mean alloc {alloc_str}")

    # Full precision, matching upload_merged_model's own load_state_dict step exactly --
    # loading directly into a 4-bit model here would silently drop/mismatch quantized keys.
    model, eval_tok = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-0.6B", max_seq_length=1024, load_in_4bit=False, dtype=torch.float16,
    )
    target_dtype = next(model.parameters()).dtype
    cast_sd = {k: v.to(target_dtype) if v.is_floating_point() else v for k, v in merged_sd.items()}
    missing, unexpected = model.load_state_dict(cast_sd, strict=False)
    if missing or unexpected:
        print(f"  [warn] missing={len(missing)} unexpected={len(unexpected)}")
    FastLanguageModel.for_inference(model)

    gsm8k_res = evaluate_gsm8k(model, eval_tok, model_name=f"adaptive-budget-{total_rank_budget}",
                                num_samples=gsm8k_samples, batch_size=4)

    del model, eval_tok, cast_sd, merged_sd
    gc.collect()
    torch.cuda.empty_cache()
    return gsm8k_res["exact_match"]

budget_results = {}
for budget in [16, 20, 24, 32, 40]:
    em = build_and_evaluate_budget(budget)
    budget_results[budget] = em
    print(f"  -> GSM8K exact match (n=40): {em:.4f}\n")

print("Summary (GSM8K EM, n=40 proxy):")
for b, em in budget_results.items():
    print(f"  budget={b:<4} GSM8K EM={em:.4f}")
print("\nBefore concluding anything, re-run the best-scoring budget with num_samples=200")
print("and add HumanEval/Dolly -- this sweep only checks GSM8K at reduced n for speed.")


budget=16: mean alloc dolly=4.99, metamath=5.39, codealpaca=5.61
==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K] Evaluating: adaptive-budget-16
────────────────────────────────────────────────────────────


adaptive-budget-16 [gsm8k]: 100%|██████████| 10/10 [01:57<00:00, 11.78s/it]


  Exact Match: 0.0000
  -> GSM8K exact match (n=40): 0.0000

budget=20: mean alloc dolly=5.91, metamath=7.12, codealpaca=6.97
==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

  Exact Match: 0.0000
  -> GSM8K exact match (n=40): 0.0000

budget=40: mean alloc dolly=12.04, metamath=14.06, codealpaca=13.91
==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K] Evaluating: adaptive-budget-40
────────────────────────────────────────────────────────────


adaptive-budget-40 [gsm8k]: 100%|██████████| 10/10 [02:00<00:00, 12.00s/it]


  Exact Match: 0.0750
  -> GSM8K exact match (n=40): 0.0750

Summary (GSM8K EM, n=40 proxy):
  budget=16   GSM8K EM=0.0000
  budget=20   GSM8K EM=0.0000
  budget=24   GSM8K EM=0.0250
  budget=32   GSM8K EM=0.0000
  budget=40   GSM8K EM=0.0750

Before concluding anything, re-run the best-scoring budget with num_samples=200
and add HumanEval/Dolly -- this sweep only checks GSM8K at reduced n for speed.
